In [16]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [18]:
IMG_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\images"
LBL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\labels"


In [19]:
class MRISliceDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, files):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img = cv2.imread(os.path.join(self.img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        img = img.astype(np.float32) / 255.0
        img = torch.tensor(img).permute(2,0,1)

        with open(os.path.join(self.lbl_dir,
                               img_name.replace(".png",".txt"))) as f:
            label = int(f.read().strip())

        return img, label


In [20]:
all_imgs = sorted(os.listdir(IMG_DIR))
labels = [int(open(os.path.join(LBL_DIR, f.replace(".png",".txt"))).read())
          for f in all_imgs]

train_files, val_files = train_test_split(
    all_imgs, test_size=0.2, random_state=42, stratify=labels
)

train_ds = MRISliceDataset(IMG_DIR, LBL_DIR, train_files)
val_ds   = MRISliceDataset(IMG_DIR, LBL_DIR, val_files)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

print("Data ready ✅")


Data ready ✅


In [ ]:
class HybridCNNViT(nn.Module):
    def __init__(self):
        super().__init__()

        # 🔹 CNN branch (STRONG)
        self.cnn = timm.create_model(
            "resnet18",
            pretrained=True,
            num_classes=0
        )

        # 🔹 ViT branch
        self.vit = timm.create_model(
            "vit_small_patch16_224",
            pretrained=True,
            num_classes=0
        )

        # Attention fusion
        self.attn = nn.Sequential(
            nn.Linear(512 + 384, 256),
            nn.ReLU(),
            nn.Linear(256, 512 + 384),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(512 + 384),
            nn.Linear(512 + 384, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        cnn_feat = self.cnn(x)
        vit_feat = self.vit(x)

        combined = torch.cat([cnn_feat, vit_feat], dim=1)
        attn_weights = self.attn(combined)
        fused = combined * attn_weights

        return self.classifier(fused)


In [ ]:
model = HybridCNNViT().to(device)


pos_weight = torch.tensor([761/627]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=1e-4
)

print("Hybrid model initialized ✅")


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

c:\Users\PRAVEEN S\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PRAVEEN S\.cache\huggingface\hub\models--timm--resnet18.a1_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Hybrid model initialized ✅


In [23]:
def train_epoch(model, loader, epoch):
    model.train()
    total_loss = 0
    for imgs, labels in tqdm(loader, desc=f"Epoch {epoch+1}"):
        imgs = imgs.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        outputs = model(imgs).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


In [24]:
def eval_epoch(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = torch.sigmoid(model(imgs)).cpu().numpy()
            preds.extend((outputs > 0.5).astype(int).flatten())
            trues.extend(labels.numpy())
    return preds, trues


In [25]:
EPOCHS = 10

for epoch in range(EPOCHS):
    loss = train_epoch(model, train_loader, epoch)
    preds, trues = eval_epoch(model, val_loader)

    print(f"\nEpoch {epoch+1} | Loss: {loss:.4f}")
    print(classification_report(trues, preds))


Epoch 1: 100%|██████████| 347/347 [33:12<00:00,  5.74s/it]



Epoch 1 | Loss: 0.3018
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       761
           1       0.99      0.90      0.95       627

    accuracy                           0.95      1388
   macro avg       0.96      0.95      0.95      1388
weighted avg       0.96      0.95      0.95      1388



Epoch 2: 100%|██████████| 347/347 [33:14<00:00,  5.75s/it]



Epoch 2 | Loss: 0.1259
              precision    recall  f1-score   support

           0       0.98      0.89      0.93       761
           1       0.88      0.97      0.92       627

    accuracy                           0.93      1388
   macro avg       0.93      0.93      0.93      1388
weighted avg       0.93      0.93      0.93      1388



Epoch 3: 100%|██████████| 347/347 [33:04<00:00,  5.72s/it]


...


Epoch 8 | Loss: 0.0302
              precision    recall  f1-score   support

           0       0.90      1.00      0.94       761
           1       1.00      0.86      0.92       627

    accuracy                           0.94      1388
   macro avg       0.95      0.93      0.93      1388
weighted avg       0.94      0.94      0.94      1388



Epoch 9: 100%|██████████| 347/347 [31:35<00:00,  5.46s/it]



Epoch 9 | Loss: 0.0200
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       761
           1       0.98      0.97      0.97       627

    accuracy                           0.97      1388
   macro avg       0.98      0.97      0.97      1388
weighted avg       0.97      0.97      0.97      1388



Epoch 10: 100%|██████████| 347/347 [33:14<00:00,  5.75s/it]



Epoch 10 | Loss: 0.0242
              precision    recall  f1-score   support

           0       0.94      0.99      0.97       761
           1       0.99      0.93      0.96       627

    accuracy                           0.96      1388
   macro avg       0.97      0.96      0.96      1388
weighted avg       0.97      0.96      0.96      1388

